# CS6501 3D Computer Vision --- Exercise 1 (coding)

Companion to the written exercise. Every place you must write code is marked `# TODO`.
Each task ends with a **self-check**: run it and make sure it prints `PASS` before moving on.

Requirements: `numpy` and `matplotlib`. Nothing else, and no downloads.

| Task | Points |
|---|---|
| A --- softmax regression by hand | 5 |
| B --- demosaicing | 8 &mdash; *separate package, see `code/p2-demosaic/`* |
| C --- building a camera | 7 |
| D --- `get_rays` | 5 |
| Demo 1 --- diffuse and specular reflection | -- |
| Demo 2 --- two-view geometry end to end | -- |

The two demos are already written. Run them and look at the output.

**Convention** (OpenCV): the camera looks down its $+z$ axis,
image $x$ is right, image $y$ is **down**, and $X_\text{cam} = R\,X_\text{world} + t$ with
$t = -R\mathbf{C}$.

In [ ]:
import os, sys, subprocess
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

# Locate the repo directories whether this notebook runs from code/ or from the repo root.
HERE = os.getcwd()
ROOT = HERE if os.path.isdir(os.path.join(HERE, "code")) else os.path.dirname(HERE)
DATA, CODE = os.path.join(ROOT, "data"), os.path.join(ROOT, "code")

if not os.path.exists(os.path.join(DATA, "two_view.npz")):
    print("generating data ...")
    subprocess.run([sys.executable, os.path.join(CODE, "make_data.py")], check=True)

print("data dir:", DATA)
print(sorted(f for f in os.listdir(DATA) if f.endswith(".npz")))

---
# Task A --- Softmax regression by hand [5 points]

Implement the three functions below following your derivation in Part 2,
Problem 1. Use the **sum** (not the mean) over training examples, to match the written derivation:

$$L = \sum_i \ell(y_i, \hat y_i) + \lambda \sum_k \lVert w_k \rVert_2^2,
\qquad \nabla_W L = X^\mathsf{T}(\hat Y - Y) + 2\lambda W.$$

Two things to watch: subtract the row max inside `softmax` for numerical stability, and do not
forget the $2\lambda W$ term.

In [ ]:
def softmax(Z):
    """Row-wise softmax. Z: (N, K) logits -> (N, K) probabilities."""
    # TODO: subtract the row-wise max for stability, then exponentiate and normalize.
    raise NotImplementedError


def cross_entropy_loss(W, X, Y, lam):
    """W: (d, K)  X: (N, d)  Y: (N, K) one-hot  ->  scalar loss, summed over examples."""
    # TODO: cross-entropy summed over examples, plus the L2 penalty lam * ||W||_F^2.
    raise NotImplementedError


def softmax_regression_grad(W, X, Y, lam):
    """Analytic gradient dL/dW, same shape as W."""
    # TODO: the gradient you derived in Part 2, Problem 1.
    raise NotImplementedError

In [ ]:
# --- SELF-CHECK A: central finite differences vs your analytic gradient ---
def grad_check(W, X, Y, lam, n_checks=60, eps=1e-5, seed=0):
    rng = np.random.default_rng(seed)
    G = softmax_regression_grad(W, X, Y, lam)
    num, ana = [], []
    for _ in range(n_checks):
        i, j = rng.integers(W.shape[0]), rng.integers(W.shape[1])
        Wp, Wm = W.copy(), W.copy()
        Wp[i, j] += eps
        Wm[i, j] -= eps
        num.append((cross_entropy_loss(Wp, X, Y, lam) - cross_entropy_loss(Wm, X, Y, lam)) / (2 * eps))
        ana.append(G[i, j])
    num, ana = np.array(num), np.array(ana)
    return np.max(np.abs(num - ana) / np.maximum(np.abs(num) + np.abs(ana), 1e-12))

rng = np.random.default_rng(0)
N, d, K = 64, 20, 5
Xc = rng.normal(size=(N, d))
Yc = np.eye(K)[rng.integers(0, K, N)]
Wc = rng.normal(scale=0.3, size=(d, K))

rel = grad_check(Wc, Xc, Yc, lam=0.1)
print(f"max relative error: {rel:.3e}")
assert rel < 1e-7, "FAIL - your analytic gradient disagrees with finite differences"
print("PASS")

---
# Task C --- Building a camera [7 points]

Implement `make_K`, `look_at` and `project`.

The self-check reuses the hand-computed numbers from Part 3, Problem 4: with
$K = [[500,0,320],[0,500,240],[0,0,1]]$, a camera at $(0,0,5)$ looking at the origin with world up
$(0,1,0)$, the world point $(1,1,0)$ must land at pixel $(420, 140)$.

In [ ]:
def make_K(f_mm, sensor_mm, img_wh):
    """Intrinsics from a physical sensor spec, principal point at the image centre."""
    # TODO: f_x = f_mm * W / sensor_width_mm, and likewise for f_y.
    raise NotImplementedError


def look_at(eye, target, up=(0.0, 1.0, 0.0)):
    """World-to-camera (R, t). Rows of R are the camera x/y/z axes; t = -R C."""
    # TODO: build the camera axes (forward z, right x, down y), stack them as the ROWS of R,
    # then set t so that X_cam = R X_world + t places the camera centre at the origin.
    raise NotImplementedError


def project(K, R, t, X):
    """X: (N, 3) world points -> (uv (N, 2), depth (N,))."""
    # TODO: world -> camera, apply K, divide by the third coordinate.
    raise NotImplementedError

In [ ]:
# --- SELF-CHECK C ---
K_chk = make_K(5.0, (6.4, 4.8), (640, 480))
assert np.allclose(K_chk, [[500, 0, 320], [0, 500, 240], [0, 0, 1]]), f"K is wrong:\n{K_chk}"

R_chk, t_chk = look_at([0, 0, 5], [0, 0, 0], [0, 1, 0])
assert np.allclose(R_chk, np.diag([1.0, -1.0, -1.0])), f"R is wrong:\n{R_chk}"
assert np.allclose(t_chk, [0, 0, 5]), f"t is wrong: {t_chk}"

uv, z = project(K_chk, R_chk, t_chk, np.array([[1.0, 1.0, 0.0]]))
assert np.allclose(uv, [[420.0, 140.0]]), f"projection is wrong: {uv}"
assert z[0] > 0, "the point should be in front of the camera"
print("PASS  (1,1,0) -> (420, 140), depth", z[0])

In [ ]:
# Render the wireframe scene from an orbiting camera.
sc = np.load(os.path.join(DATA, "scene_wireframe.npz"))
CUBE, GRID = sc["cube_vertices"], sc["grid_segments"]
EDGE_GROUPS = [(sc["cube_edges_x"], "C1"), (sc["cube_edges_z"], "C2"), (sc["cube_edges_y"], "C0")]
IMG_WH = (640, 480)
K = make_K(5.0, (6.4, 4.8), IMG_WH)


def draw_view(ax, R, t, title=""):
    for seg in GRID:
        uv, z = project(K, R, t, seg)
        if np.all(z > 0):
            ax.plot(uv[:, 0], uv[:, 1], color="0.85", lw=0.8, zorder=1)
    uv, _ = project(K, R, t, CUBE)
    for edges, col in EDGE_GROUPS:
        for i, j in edges:
            ax.plot(uv[[i, j], 0], uv[[i, j], 1], color=col, lw=2, zorder=2)
    ax.set_xlim(0, IMG_WH[0]); ax.set_ylim(IMG_WH[1], 0)      # v increases downward
    ax.set_aspect("equal"); ax.set_title(title, fontsize=9)


fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, ang in zip(axes, [20, 55, 90]):
    a = np.deg2rad(ang)
    R, t = look_at([7 * np.sin(a), 3.0, 7 * np.cos(a)], [0, 0.5, 0])
    draw_view(ax, R, t, f"orbit {ang} deg")
plt.tight_layout(); plt.show()
print("The orange and green edge families are each parallel in 3D. Extend one family in your head:")
print("it converges to that direction's vanishing point (Part 3, Problem 2), and both vanishing")
print("points lie on the ground plane's horizon.")

---
# Task D --- `get_rays` [5 points]

Back-project pixels to world-space rays, following Part 3, Problem 5. This is the function every
NeRF implementation starts with, and it is why Problem 4(d) makes such a fuss about the
camera-to-world convention.

The self-check samples random pixels, walks a random $\lambda > 0$ along each ray, and reprojects;
you must land back on the pixel you started from.

In [ ]:
def c2w_from_Rt(R, t):
    """4x4 camera-to-world matrix from world-to-camera (R, t)."""
    # TODO: see Part 3, Problem 4(d). The rotation block is R^T and the translation is the
    # camera centre C = -R^T t.
    raise NotImplementedError


def get_rays(K, c2w, uv):
    """uv: (N, 2) pixels -> (origins (N, 3), directions (N, 3)) in world coordinates."""
    # TODO: o = camera centre; d = R_c2w @ K^-1 @ [u, v, 1]^T.
    # Decide whether to normalize d, and be clear what that means for the units of lambda.
    raise NotImplementedError

In [ ]:
# --- SELF-CHECK D ---
rng = np.random.default_rng(1)
R, t = look_at([4.0, 3.0, 6.0], [0, 0, 0])
c2w = c2w_from_Rt(R, t)
assert np.allclose(c2w[:3, 3], -R.T @ t), "c2w translation should be the camera centre"

uv = rng.uniform([0, 0], [640, 480], size=(500, 2))
o, d = get_rays(K, c2w, uv)
lam = rng.uniform(0.5, 50.0, size=(500, 1))
uv_re, depth = project(K, R, t, o + lam * d)

err = np.abs(uv_re - uv).max()
print(f"max reprojection error over 500 random rays: {err:.3e} px")
assert err < 1e-6, "FAIL - points along the ray do not reproject to the original pixel"
assert np.all(depth > 0), "FAIL - sampled points should be in front of the camera"
print("PASS")

---
# Demo 1 --- diffuse and specular reflection

Nothing to implement. A Lambertian surface is said to "look identical at all views"; the rightmost
panel below is the precise version of that claim.

In [ ]:
# A sphere of radius 1 under orthographic viewing: each visible point's normal is (x, y, z).
n_px = 256
gx, gy = np.meshgrid(np.linspace(-1, 1, n_px), np.linspace(1, -1, n_px))
inside = gx ** 2 + gy ** 2 <= 1
gz = np.sqrt(np.clip(1 - gx ** 2 - gy ** 2, 0, None))
Nrm = np.stack([gx, gy, gz], -1)

light = np.array([-0.4, 0.6, 0.7]); light /= np.linalg.norm(light)
view = np.array([0.0, 0.0, 1.0])
ndotl = np.clip(Nrm @ light, 0, None)
refl = 2 * (Nrm @ light)[..., None] * Nrm - light          # mirror direction

fig, axes = plt.subplots(1, 4, figsize=(14, 3.8))
axes[0].imshow(np.where(inside, ndotl, 0), cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Lambertian:  $I=\\rho\\,(n\\cdot l)$", fontsize=9)
for ax, p in zip(axes[1:3], [10, 60]):
    spec = np.clip(refl @ view, 0, None) ** p
    ax.imshow(np.where(inside, 0.15 * ndotl + spec, 0), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"Phong specular, exponent {p}", fontsize=9)
for ax in axes[:3]:
    ax.axis("off")

# The claim made precise: fix ONE patch and one light, then sweep the VIEWING direction.
ax = axes[3]
n_ = np.array([0.0, 0.0, 1.0])
l_ = np.array([np.sin(np.deg2rad(40)), 0.0, np.cos(np.deg2rad(40))])
r_ = 2 * (n_ @ l_) * n_ - l_
ang = np.linspace(-89, 89, 400)
v_ = np.stack([np.sin(np.deg2rad(ang)), np.zeros_like(ang), np.cos(np.deg2rad(ang))], -1)
ax.plot(ang, np.full_like(ang, n_ @ l_), label="Lambertian")
for p in (10, 60):
    ax.plot(ang, np.clip(v_ @ r_, 0, None) ** p, label=f"Phong n={p}")
ax.set_xlabel("viewing angle (deg)"); ax.set_ylabel("observed brightness")
ax.set_title("one patch, fixed light, varying view", fontsize=9)
ax.legend(fontsize=7); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("Right panel: the Lambertian curve is FLAT -- that patch is equally bright from every")
print("direction -- 'looks identical at all views', i.e. Part 4, Problem 4(a).")
print("The Phong lobe is not flat. That is why the highlight moves as you walk around a shiny")
print("object, and why photometric stereo's linear system breaks on it (Problem 4c).")

---
# Demo 2 --- two-view geometry, end to end

Nothing to implement. The two views were generated with the camera model you wrote
in Task C, so $R$, $t$, $E$ and $F$ are known **exactly** and every estimate can be scored against
truth rather than eyeballed.

In [ ]:
tv = np.load(os.path.join(DATA, "two_view.npz"))
X_gt, x1, x2 = tv["X"], tv["x1"], tv["x2"]
K1, K2 = tv["K1"], tv["K2"]
R_rel_gt, t_rel_gt, F_gt = tv["R_rel"], tv["t_rel"], tv["F"]
print(f"{len(X_gt)} correspondences, baseline {np.linalg.norm(t_rel_gt):.3f}")


def normalize_points(x):
    """Hartley normalization: centroid to the origin, mean distance sqrt(2)."""
    c = x.mean(axis=0)
    s = np.sqrt(2) / np.linalg.norm(x - c, axis=1).mean()
    T = np.array([[s, 0, -s * c[0]], [0, s, -s * c[1]], [0, 0, 1.0]])
    return (np.c_[x, np.ones(len(x))] @ T.T)[:, :2], T


def eight_point(a, b, normalize=True, return_diag=False):
    if normalize:
        an, T1 = normalize_points(a)
        bn, T2 = normalize_points(b)
    else:
        an, bn, T1, T2 = a, b, np.eye(3), np.eye(3)

    u1, v1, u2, v2 = an[:, 0], an[:, 1], bn[:, 0], bn[:, 1]
    A = np.stack([u2 * u1, u2 * v1, u2, v2 * u1, v2 * v1, v2,
                  u1, v1, np.ones_like(u1)], axis=1)
    F = np.linalg.svd(A)[2][-1].reshape(3, 3)

    U, S, Vt = np.linalg.svd(F)          # enforce rank 2, BEFORE undoing the normalization
    F = U @ np.diag([S[0], S[1], 0]) @ Vt
    F = T2.T @ F @ T1
    F /= np.linalg.norm(F)

    if return_diag:
        col = np.linalg.norm(A, axis=0)
        return F, {"col_ratio": col.max() / col.min()}
    return F


def sym_epipolar_distance(F, a, b):
    ah, bh = np.c_[a, np.ones(len(a))], np.c_[b, np.ones(len(b))]
    l2, l1 = ah @ F.T, bh @ F
    num = np.abs(np.sum(bh * l2, axis=1))
    return np.mean(0.5 * (num / np.linalg.norm(l2[:, :2], axis=1)
                          + num / np.linalg.norm(l1[:, :2], axis=1)))


F_hat = eight_point(x1, x2)
print(f"||F_hat - F_gt|| = {min(np.linalg.norm(F_hat - F_gt), np.linalg.norm(F_hat + F_gt)):.3e}")

rng = np.random.default_rng(3)
print(f"\n{'points':>12s} {'normalized':>11s} {'col ratio':>12s} {'epi dist (px)':>15s}")
for label, (a, b) in [("clean", (x1, x2)),
                      ("sigma=0.5px", (x1 + rng.normal(0, .5, x1.shape),
                                       x2 + rng.normal(0, .5, x2.shape)))]:
    for norm in (True, False):
        Fe, diag = eight_point(a, b, normalize=norm, return_diag=True)
        print(f"{label:>12s} {str(norm):>11s} {diag['col_ratio']:12.3e} "
              f"{sym_epipolar_distance(Fe, x1, x2):15.4f}")
print("\nOn clean points both are exact -- an exact null vector exists regardless of conditioning.")
print("Under noise, normalization is worth a large factor. That is Part 5's design-matrix argument.")

In [ ]:
# Epipolar lines must form a pencil through the epipole (Part 5, Problem 1c).
def draw_epipolar(ax, F, x_from, x_on, which, n=8, wh=(640, 480)):
    idx = np.linspace(0, len(x_from) - 1, n).astype(int)
    lines = np.c_[x_from[idx], np.ones(n)] @ (F.T if which == 2 else F)
    uu = np.array([-4000.0, 4000.0])
    cols = plt.cm.tab10(np.linspace(0, 1, n))
    for l, c in zip(lines, cols):
        ax.plot(uu, -(l[0] * uu + l[2]) / l[1], color=c, lw=0.9)
    ax.scatter(x_on[idx, 0], x_on[idx, 1], s=30, c=cols, edgecolors="k", linewidths=.5, zorder=5)
    ax.set_xlim(0, wh[0]); ax.set_ylim(wh[1], 0); ax.set_aspect("equal")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
draw_epipolar(axes[0], F_hat, x2, x1, which=1); axes[0].set_title("view 1", fontsize=9)
draw_epipolar(axes[1], F_hat, x1, x2, which=2); axes[1].set_title("view 2", fontsize=9)
plt.tight_layout(); plt.show()

e1 = np.linalg.svd(F_hat)[2][-1]; e2 = np.linalg.svd(F_hat.T)[2][-1]
print("epipole in view 1:", e1[:2] / e1[2], "   view 2:", e2[:2] / e2[2])
print("Both lie outside the image here -- normal, since each camera is outside the other's view.")

In [ ]:
# Essential matrix -> four candidate poses -> cheirality -> triangulation.
def decompose_essential(E):
    U, S, Vt = np.linalg.svd(E)
    if np.linalg.det(U) < 0:
        U[:, -1] *= -1
    if np.linalg.det(Vt) < 0:
        Vt[-1, :] *= -1
    Wm = np.array([[0.0, -1, 0], [1, 0, 0], [0, 0, 1]])
    Ra, Rb, tt = U @ Wm @ Vt, U @ Wm.T @ Vt, U[:, 2]
    return [(Ra, tt), (Ra, -tt), (Rb, tt), (Rb, -tt)]


def triangulate_dlt(P1, P2, a, b):
    X = np.empty((len(a), 3))
    for i, ((u1, v1), (u2, v2)) in enumerate(zip(a, b)):
        A = np.stack([u1 * P1[2] - P1[0], v1 * P1[2] - P1[1],
                      u2 * P2[2] - P2[0], v2 * P2[2] - P2[1]])
        Xh = np.linalg.svd(A)[2][-1]
        X[i] = Xh[:3] / Xh[3]
    return X


def align_similarity(src, dst):
    """Umeyama: the scale, rotation and translation minimizing ||s R src + t - dst||^2."""
    mu_s, mu_d = src.mean(0), dst.mean(0)
    S_, D_ = src - mu_s, dst - mu_d
    U, sv, Vt = np.linalg.svd(D_.T @ S_ / len(src))
    Wm = np.eye(3)
    if np.linalg.det(U) * np.linalg.det(Vt) < 0:
        Wm[2, 2] = -1
    Rr = U @ Wm @ Vt
    s = np.trace(np.diag(sv) @ Wm) / ((S_ ** 2).sum() / len(src))
    return s, Rr, mu_d - s * Rr @ mu_s


E_hat = K2.T @ F_hat @ K1
U, S, Vt = np.linalg.svd(E_hat)
E_hat = U @ np.diag([1.0, 1.0, 0.0]) @ Vt          # project onto the essential manifold

P1 = K1 @ np.hstack([np.eye(3), np.zeros((3, 1))])
cands = decompose_essential(E_hat)
counts = []
for Rc, tc in cands:
    Xc = triangulate_dlt(P1, K2 @ np.hstack([Rc, tc.reshape(3, 1)]), x1, x2)
    counts.append(int(np.sum((Xc[:, 2] > 0) & ((Xc @ Rc.T + tc)[:, 2] > 0))))
print(f"cheirality counts for the four candidates: {counts}  (of {len(x1)} points)")
print("Exactly one candidate puts every point in front of BOTH cameras -- Part 3, Problem 5(c).")

R_est, t_est = cands[int(np.argmax(counts))]
rot_err = np.degrees(np.arccos(np.clip((np.trace(R_est @ R_rel_gt.T) - 1) / 2, -1, 1)))
t_err = np.degrees(np.arccos(np.clip(abs(t_est @ (t_rel_gt / np.linalg.norm(t_rel_gt))), -1, 1)))
print(f"\nrotation error {rot_err:.3e} deg,  translation direction error {t_err:.3e} deg")

P2 = K2 @ np.hstack([R_est, t_est.reshape(3, 1)])
X_est = triangulate_dlt(P1, P2, x1, x2)
X_gt_cam1 = X_gt @ tv["R1"].T + tv["t1"]
s, Ra_, ta_ = align_similarity(X_est, X_gt_cam1)
err3d = np.linalg.norm((s * (X_est @ Ra_.T) + ta_) - X_gt_cam1, axis=1).mean()
print(f"mean 3D error after similarity alignment: {err3d:.3e}")
print(f"recovered scale {s:.4f}  vs true baseline {np.linalg.norm(t_rel_gt):.4f}")
print("They match -- and that is the point: two views fix the geometry only up to scale, so the")
print("alignment had to supply the one number the images could never contain.")

fig = plt.figure(figsize=(11, 4.2))
for i, (P, ttl) in enumerate([(X_gt_cam1, "ground truth"),
                              (s * (X_est @ Ra_.T) + ta_, "triangulated, aligned")]):
    ax = fig.add_subplot(1, 2, i + 1, projection="3d")
    ax.scatter(P[:, 0], P[:, 1], P[:, 2], s=10, c=P[:, 2], cmap="viridis")
    ax.set_title(ttl, fontsize=9)
plt.tight_layout(); plt.show()